# SaaS Product Analytics

**Product:** Simulated B2B SaaS application  
**Period:** January 2023 – December 2023  
**Tools:** Python, Pandas, Matplotlib, Seaborn  
**Author:** [Your Name]

This analysis covers the core product and revenue metrics used by data analysts in SaaS companies: user activity, retention cohorts, funnel analysis, churn, and monthly recurring revenue.

---

## Table of Contents
1. [Data Generation](#1)
2. [User Growth — MAU and DAU](#2)
3. [Retention Cohort Analysis](#3)
4. [Feature Adoption Funnel](#4)
5. [Churn Analysis](#5)
6. [Revenue — MRR and ARR](#6)
7. [Summary and Findings](#7)

## 1. Data Generation <a id='1'></a>

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
import os

warnings.filterwarnings('ignore')
os.makedirs('plots', exist_ok=True)

plt.rcParams.update({
    'figure.dpi':        130,
    'font.family':       'DejaVu Sans',
    'axes.spines.top':   False,
    'axes.spines.right': False,
    'axes.grid':         True,
    'grid.alpha':        0.25,
    'grid.linestyle':    '--',
    'axes.facecolor':    '#F8F9FA',
    'figure.facecolor':  'white',
})

C1 = '#2C3E50'  # dark slate
C2 = '#2E86C1'  # blue
C3 = '#1E8449'  # green
C4 = '#C0392B'  # red
C5 = '#D4820A'  # amber
C6 = '#7D3C98'  # purple

print('Libraries loaded.')

In [ ]:
np.random.seed(42)

N_USERS = 3000
START   = pd.Timestamp('2023-01-01')
END     = pd.Timestamp('2023-12-31')

# -- Users table --
signup_dates = pd.to_datetime(
    np.random.choice(pd.date_range(START, END), N_USERS)
).sort_values()

plans = np.random.choice(
    ['free', 'starter', 'pro', 'enterprise'],
    N_USERS, p=[0.40, 0.30, 0.20, 0.10]
)
plan_price = {'free': 0, 'starter': 29, 'pro': 79, 'enterprise': 299}

users = pd.DataFrame({
    'user_id':     range(1, N_USERS + 1),
    'signup_date': signup_dates,
    'plan':        plans,
    'mrr':         [plan_price[p] for p in plans],
})

# Churn: ~15% of paid users churn within the year
churned_mask = (
    (users['plan'] != 'free') &
    (np.random.rand(N_USERS) < 0.15)
)
churn_offset = pd.to_timedelta(
    np.random.randint(30, 300, N_USERS), unit='D'
)
users['churn_date'] = None
users.loc[churned_mask, 'churn_date'] = (
    users.loc[churned_mask, 'signup_date'] + churn_offset[churned_mask]
).clip(upper=END)

# -- Events table --
features = ['dashboard_view', 'report_created', 'integration_connected',
            'export_used', 'team_member_invited']

event_rows = []
for _, u in users.iterrows():
    end_date = u['churn_date'] if pd.notna(u['churn_date']) else END
    active_days = max((end_date - u['signup_date']).days, 1)
    n_events = int(np.random.exponential(scale=max(active_days * 0.4, 1)))
    n_events = min(n_events, 600)
    if n_events == 0:
        continue
    event_dates = u['signup_date'] + pd.to_timedelta(
        np.random.randint(0, active_days, n_events), unit='D'
    )
    # Feature adoption probabilities differ by plan
    probs = {
        'free':       [0.60, 0.10, 0.05, 0.05, 0.20],
        'starter':    [0.40, 0.20, 0.15, 0.10, 0.15],
        'pro':        [0.30, 0.25, 0.20, 0.15, 0.10],
        'enterprise': [0.25, 0.25, 0.20, 0.20, 0.10],
    }[u['plan']]
    chosen_features = np.random.choice(features, n_events, p=probs)
    for d, f in zip(event_dates, chosen_features):
        event_rows.append({'user_id': u['user_id'], 'event_date': d, 'feature': f})

events = pd.DataFrame(event_rows)
events['event_date'] = pd.to_datetime(events['event_date'])
events = events[events['event_date'] <= END].reset_index(drop=True)

# Save CSVs
users.to_csv('saas_users.csv', index=False)
events.to_csv('saas_events.csv', index=False)

print(f'Users:  {len(users):,}')
print(f'Events: {len(events):,}')
users.head()

## 2. User Growth — MAU and DAU <a id='2'></a>

Monthly Active Users (MAU) counts distinct users who triggered at least one event in a given month. DAU/MAU ratio (stickiness) measures how often active users return within a month — a ratio above 20% is generally considered healthy for SaaS.

In [ ]:
events['month'] = events['event_date'].dt.to_period('M')
events['date']  = events['event_date'].dt.date

mau = events.groupby('month')['user_id'].nunique().reset_index()
mau.columns = ['month', 'mau']
mau['month_str'] = mau['month'].astype(str)

dau = events.groupby('date')['user_id'].nunique().reset_index()
dau.columns = ['date', 'dau']
dau['date'] = pd.to_datetime(dau['date'])
dau['month'] = dau['date'].dt.to_period('M')

avg_dau = dau.groupby('month')['dau'].mean().reset_index()
avg_dau.columns = ['month', 'avg_dau']

mau = mau.merge(avg_dau, on='month')
mau['dau_mau_ratio'] = (mau['avg_dau'] / mau['mau']).round(3)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('User Activity — 2023', fontsize=13, fontweight='bold')

axes[0].bar(mau['month_str'], mau['mau'], color=C2, edgecolor='white', width=0.6)
axes[0].plot(mau['month_str'], mau['mau'], color=C1, marker='o',
             markersize=5, linewidth=1.5)
axes[0].set_title('Monthly Active Users (MAU)', fontweight='bold')
axes[0].set_xlabel('Month')
axes[0].set_ylabel('Unique Active Users')
axes[0].tick_params(axis='x', rotation=45)
for i, v in enumerate(mau['mau']):
    axes[0].text(i, v + 8, str(v), ha='center', fontsize=8, color=C1)

axes[1].bar(mau['month_str'], mau['dau_mau_ratio'] * 100,
            color=C3, edgecolor='white', width=0.6)
axes[1].axhline(20, color=C4, linewidth=1.5, linestyle='--', label='20% benchmark')
axes[1].set_title('DAU / MAU Ratio (Stickiness)', fontweight='bold')
axes[1].set_xlabel('Month')
axes[1].set_ylabel('DAU / MAU (%)')
axes[1].yaxis.set_major_formatter(mticker.PercentFormatter())
axes[1].tick_params(axis='x', rotation=45)
axes[1].legend(fontsize=9)

plt.tight_layout()
plt.savefig('plots/01_mau_dau.png', bbox_inches='tight')
plt.show()

print('MAU summary:')
print(mau[['month_str','mau','avg_dau','dau_mau_ratio']].to_string(index=False))

## 3. Retention Cohort Analysis <a id='3'></a>

Cohort analysis groups users by signup month and tracks what percentage return in each subsequent month. This is one of the most important charts in product analytics — it shows whether the product retains users over time or whether it leaks.

In [ ]:
users['cohort'] = users['signup_date'].dt.to_period('M')
events_merged   = events.merge(users[['user_id','cohort']], on='user_id')

events_merged['period_number'] = (
    events_merged['month'].astype(int) - events_merged['cohort'].astype(int)
)

cohort_data = (
    events_merged
    .groupby(['cohort','period_number'])['user_id']
    .nunique()
    .reset_index()
)
cohort_sizes = users.groupby('cohort')['user_id'].count()
cohort_data  = cohort_data.merge(
    cohort_sizes.rename('cohort_size'), on='cohort'
)
cohort_data['retention'] = cohort_data['user_id'] / cohort_data['cohort_size']

cohort_pivot = cohort_data.pivot_table(
    index='cohort', columns='period_number', values='retention'
)
cohort_pivot = cohort_pivot.iloc[:, :9]   # periods 0–8
cohort_pivot.index = cohort_pivot.index.astype(str)
cohort_pivot.columns = [f'M+{c}' for c in cohort_pivot.columns]

fig, ax = plt.subplots(figsize=(13, 7))
sns.heatmap(
    cohort_pivot,
    ax=ax,
    annot=True,
    fmt='.0%',
    cmap='Blues',
    vmin=0, vmax=1,
    linewidths=0.5,
    linecolor='white',
    cbar_kws={'label': 'Retention Rate', 'shrink': 0.8}
)
ax.set_title('Monthly Retention by Signup Cohort', fontsize=13, fontweight='bold', pad=12)
ax.set_xlabel('Months Since Signup', fontweight='bold')
ax.set_ylabel('Signup Cohort', fontweight='bold')
plt.tight_layout()
plt.savefig('plots/02_retention_cohort.png', bbox_inches='tight')
plt.show()

print('Average retention by period:')
print(cohort_pivot.mean().apply(lambda x: f'{x:.1%}').to_string())

## 4. Feature Adoption Funnel <a id='4'></a>

The funnel tracks how many users progress through core product actions. Drop-off at each step reveals where the product loses users and where onboarding or UX improvements would have the most impact.

In [ ]:
funnel_steps = [
    ('Signed Up',              len(users)),
    ('Viewed Dashboard',       events[events['feature']=='dashboard_view']['user_id'].nunique()),
    ('Created a Report',       events[events['feature']=='report_created']['user_id'].nunique()),
    ('Connected Integration',  events[events['feature']=='integration_connected']['user_id'].nunique()),
    ('Exported Data',          events[events['feature']=='export_used']['user_id'].nunique()),
    ('Invited a Team Member',  events[events['feature']=='team_member_invited']['user_id'].nunique()),
]

funnel_df = pd.DataFrame(funnel_steps, columns=['step', 'users'])
funnel_df['pct_of_total']  = funnel_df['users'] / funnel_df['users'].iloc[0]
funnel_df['pct_of_prev']   = funnel_df['users'] / funnel_df['users'].shift(1)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('Feature Adoption Funnel', fontsize=13, fontweight='bold')

colors = [C2 if i == 0 else (C3 if v > 0.5 else (C5 if v > 0.25 else C4))
          for i, v in enumerate(funnel_df['pct_of_total'])]

bars = axes[0].barh(funnel_df['step'][::-1], funnel_df['users'][::-1],
                    color=colors[::-1], edgecolor='white', height=0.55)
for bar, val, pct in zip(bars, funnel_df['users'][::-1], funnel_df['pct_of_total'][::-1]):
    axes[0].text(bar.get_width() + 20, bar.get_y() + bar.get_height()/2,
                 f'{val:,}  ({pct:.0%})', va='center', fontsize=9, color=C1)
axes[0].set_title('Users Reaching Each Step', fontweight='bold')
axes[0].set_xlabel('Number of Users')
axes[0].set_xlim(0, funnel_df['users'].max() * 1.3)

drop = funnel_df['pct_of_prev'].fillna(1)
drop_colors = [C3 if v >= 0.6 else (C5 if v >= 0.4 else C4) for v in drop]
axes[1].bar(range(len(funnel_df)), drop * 100, color=drop_colors,
            edgecolor='white', width=0.55)
axes[1].set_xticks(range(len(funnel_df)))
axes[1].set_xticklabels(
    [s[:18] for s in funnel_df['step']], rotation=30, ha='right', fontsize=8
)
axes[1].yaxis.set_major_formatter(mticker.PercentFormatter())
axes[1].axhline(100, color='#AAAAAA', linewidth=0.8)
axes[1].set_title('Conversion from Previous Step', fontweight='bold')
axes[1].set_ylabel('Conversion Rate (%)')
for i, v in enumerate(drop):
    axes[1].text(i, v*100 + 1.5, f'{v:.0%}', ha='center', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.savefig('plots/03_funnel.png', bbox_inches='tight')
plt.show()

print(funnel_df.to_string(index=False))

## 5. Churn Analysis <a id='5'></a>

Churn rate is the percentage of paying customers who cancel in a given month. Monthly churn above 5% is a warning sign for a SaaS business. We also look at which plan churns most.

In [ ]:
paid = users[users['plan'] != 'free'].copy()
paid['churn_date'] = pd.to_datetime(paid['churn_date'])
paid['churn_month'] = paid['churn_date'].dt.to_period('M')

monthly_churned = (
    paid[paid['churn_date'].notna()]
    .groupby('churn_month')['user_id'].count()
    .reset_index()
)
monthly_churned.columns = ['month', 'churned']

# Active at start of each month
months = pd.period_range('2023-01', '2023-12', freq='M')
active_counts = []
for m in months:
    m_start = m.to_timestamp()
    active = paid[
        (paid['signup_date'] < m_start) &
        ((paid['churn_date'].isna()) | (paid['churn_date'] >= m_start))
    ]
    active_counts.append({'month': m, 'active_start': len(active)})

active_df = pd.DataFrame(active_counts)
churn_df  = active_df.merge(monthly_churned, on='month', how='left').fillna(0)
churn_df['churn_rate'] = churn_df['churned'] / churn_df['active_start'].replace(0, np.nan)
churn_df['month_str']  = churn_df['month'].astype(str)

# Churn by plan
churn_by_plan = paid[paid['churn_date'].notna()].groupby('plan')['user_id'].count()
total_by_plan = paid.groupby('plan')['user_id'].count()
churn_rate_plan = (churn_by_plan / total_by_plan).fillna(0)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Churn Analysis — Paid Users', fontsize=13, fontweight='bold')

axes[0].bar(churn_df['month_str'], churn_df['churn_rate'] * 100,
            color=C4, edgecolor='white', width=0.6, alpha=0.85)
axes[0].axhline(5, color=C1, linewidth=1.5, linestyle='--', label='5% warning line')
axes[0].set_title('Monthly Churn Rate', fontweight='bold')
axes[0].set_xlabel('Month')
axes[0].set_ylabel('Churn Rate (%)')
axes[0].yaxis.set_major_formatter(mticker.PercentFormatter())
axes[0].tick_params(axis='x', rotation=45)
axes[0].legend(fontsize=9)

plan_order = ['starter', 'pro', 'enterprise']
plan_colors = [C5, C2, C6]
axes[1].bar(
    plan_order,
    [churn_rate_plan.get(p, 0) * 100 for p in plan_order],
    color=plan_colors, edgecolor='white', width=0.5
)
axes[1].set_title('Churn Rate by Plan', fontweight='bold')
axes[1].set_xlabel('Plan')
axes[1].set_ylabel('Churn Rate (%)')
axes[1].yaxis.set_major_formatter(mticker.PercentFormatter())
for i, p in enumerate(plan_order):
    v = churn_rate_plan.get(p, 0) * 100
    axes[1].text(i, v + 0.3, f'{v:.1f}%', ha='center', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.savefig('plots/04_churn.png', bbox_inches='tight')
plt.show()

print(f'Overall paid churn rate: {paid["churn_date"].notna().mean():.1%}')
print('Churn rate by plan:')
for p in plan_order:
    print(f'  {p:12s}  {churn_rate_plan.get(p, 0):.1%}')

## 6. Revenue — MRR and ARR <a id='6'></a>

Monthly Recurring Revenue (MRR) is the primary health metric for a SaaS business. We track new MRR from signups, churned MRR from cancellations, and net MRR growth each month.

In [ ]:
months = pd.period_range('2023-01', '2023-12', freq='M')
mrr_rows = []

for m in months:
    m_start = m.to_timestamp()
    m_end   = (m + 1).to_timestamp() - pd.Timedelta(days=1)

    new_users = paid[
        (paid['signup_date'] >= m_start) & (paid['signup_date'] <= m_end)
    ]
    churned_users = paid[
        (paid['churn_date'].notna()) &
        (paid['churn_date'] >= m_start) & (paid['churn_date'] <= m_end)
    ]
    active_users = paid[
        (paid['signup_date'] <= m_end) &
        ((paid['churn_date'].isna()) | (paid['churn_date'] > m_end))
    ]

    mrr_rows.append({
        'month':      m,
        'month_str':  str(m),
        'total_mrr':  active_users['mrr'].sum(),
        'new_mrr':    new_users['mrr'].sum(),
        'churned_mrr': churned_users['mrr'].sum(),
        'active_paying': len(active_users),
    })

mrr_df = pd.DataFrame(mrr_rows)
mrr_df['net_new_mrr'] = mrr_df['new_mrr'] - mrr_df['churned_mrr']
mrr_df['arr'] = mrr_df['total_mrr'] * 12

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Revenue Metrics — MRR and ARR 2023', fontsize=13, fontweight='bold')

# Total MRR
axes[0,0].fill_between(mrr_df['month_str'], mrr_df['total_mrr'],
                        color=C2, alpha=0.2)
axes[0,0].plot(mrr_df['month_str'], mrr_df['total_mrr'],
               color=C2, linewidth=2.5, marker='o', markersize=5)
axes[0,0].set_title('Total MRR', fontweight='bold')
axes[0,0].set_ylabel('MRR ($)')
axes[0,0].yaxis.set_major_formatter(mticker.StrMethodFormatter('${x:,.0f}'))
axes[0,0].tick_params(axis='x', rotation=45)

# New vs Churned MRR
x = range(len(mrr_df))
axes[0,1].bar(x, mrr_df['new_mrr'], color=C3, label='New MRR',
              edgecolor='white', width=0.4, align='center')
axes[0,1].bar([i + 0.4 for i in x], mrr_df['churned_mrr'], color=C4,
              label='Churned MRR', edgecolor='white', width=0.4, align='center')
axes[0,1].set_title('New vs Churned MRR', fontweight='bold')
axes[0,1].set_ylabel('MRR ($)')
axes[0,1].set_xticks([i+0.2 for i in x])
axes[0,1].set_xticklabels(mrr_df['month_str'], rotation=45, ha='right', fontsize=8)
axes[0,1].yaxis.set_major_formatter(mticker.StrMethodFormatter('${x:,.0f}'))
axes[0,1].legend(fontsize=9)

# Net new MRR
net_colors = [C3 if v >= 0 else C4 for v in mrr_df['net_new_mrr']]
axes[1,0].bar(mrr_df['month_str'], mrr_df['net_new_mrr'],
              color=net_colors, edgecolor='white', width=0.6)
axes[1,0].axhline(0, color=C1, linewidth=0.8)
axes[1,0].set_title('Net New MRR', fontweight='bold')
axes[1,0].set_ylabel('Net MRR ($)')
axes[1,0].yaxis.set_major_formatter(mticker.StrMethodFormatter('${x:,.0f}'))
axes[1,0].tick_params(axis='x', rotation=45)

# ARR
axes[1,1].fill_between(mrr_df['month_str'], mrr_df['arr'],
                        color=C6, alpha=0.2)
axes[1,1].plot(mrr_df['month_str'], mrr_df['arr'],
               color=C6, linewidth=2.5, marker='o', markersize=5)
axes[1,1].set_title('Annualised Recurring Revenue (ARR)', fontweight='bold')
axes[1,1].set_ylabel('ARR ($)')
axes[1,1].yaxis.set_major_formatter(mticker.StrMethodFormatter('${x:,.0f}'))
axes[1,1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig('plots/05_revenue.png', bbox_inches='tight')
plt.show()

print(f'Dec MRR:  ${mrr_df["total_mrr"].iloc[-1]:,.0f}')
print(f'Dec ARR:  ${mrr_df["arr"].iloc[-1]:,.0f}')
print(f'MRR growth Jan-Dec: {(mrr_df["total_mrr"].iloc[-1]/mrr_df["total_mrr"].iloc[0]-1):.1%}')

## 7. Summary and Findings <a id='7'></a>

In [ ]:
total_churned   = paid['churn_date'].notna().sum()
overall_churn   = total_churned / len(paid)
avg_mau         = mau['mau'].mean()
avg_stickiness  = mau['dau_mau_ratio'].mean()
m1_retention    = cohort_pivot['M+1'].mean()
biggest_drop    = funnel_df.loc[funnel_df['pct_of_prev'].idxmin()]

print('=' * 55)
print('   SAAS PRODUCT ANALYTICS — KEY FINDINGS')
print('=' * 55)
print(f'  Total users            : {len(users):,}')
print(f'  Total events           : {len(events):,}')
print(f'  Avg MAU                : {avg_mau:,.0f}')
print(f'  Avg stickiness (DAU/MAU): {avg_stickiness:.1%}')
print(f'  Avg Month-1 retention  : {m1_retention:.1%}')
print(f'  Overall paid churn     : {overall_churn:.1%}')
print(f'  Dec MRR                : ${mrr_df["total_mrr"].iloc[-1]:,.0f}')
print(f'  Dec ARR                : ${mrr_df["arr"].iloc[-1]:,.0f}')
print(f'  Biggest funnel drop    : {biggest_drop["step"]} ({biggest_drop["pct_of_prev"]:.0%})')
print()
print('  Recommendations:')
print(f'  - Focus onboarding effort on "{biggest_drop["step"]}" step')
print(f'    which has the steepest drop-off in the funnel.')
print(f'  - Stickiness is {avg_stickiness:.1%} — target is above 20% for healthy SaaS.')
print(f'  - Monitor Month-1 retention; lifting it by 5pp typically')
print(f'    has the largest impact on long-term LTV.')
print('=' * 55)